In [1]:


"""
============================================================================
 GOLD — PART B (CORRECTED): ONE-STEP-AHEAD WALK-FORWARD COMPARISON
 Random Forest + XGBoost + LSTM vs the classical models.
============================================================================

WHAT CHANGED AND WHY (read this — it's the key fix):
  The earlier version forecast all ~308 test days at once from the end of
  training (a "recursive multi-step" forecast). Over 308 blind steps the
  errors are dominated by accumulated DRIFT, not real skill, which made one
  model look artificially 4x better. This is a known pitfall.

  This version uses ONE-STEP-AHEAD (walk-forward / rolling-origin) evaluation,
  the standard approach in Hyndman & Athanasopoulos, "Forecasting: Principles
  and Practice" (time series cross-validation). Each test day:
     1. predict ONLY the next day
     2. then reveal the true value
     3. move forward one day and repeat
  This measures next-step accuracy given real recent data, and removes the
  long-horizon drift artifact.

FAIRNESS (unchanged): same data, same 80/20 split from R, same log scale,
  same test days, same metrics, plus a Diebold-Mariano test.

TO RUN (in a terminal, after installing once):
  pip install pandas numpy scikit-learn xgboost tensorflow scipy
  python gold_ml_partB.py
============================================================================
"""

import os, random
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from scipy.stats import norm

os.environ["PYTHONHASHSEED"] = "42"
random.seed(42); np.random.seed(42)

N_LAGS = 20   # previous 20 days used as predictors

# ---------------------------------------------------------------------------
# 1. LOAD data and the SAME split R used
# ---------------------------------------------------------------------------
df = (pd.read_csv("gold_clean.csv", parse_dates=["Date"])
        .sort_values("Date").reset_index(drop=True))
price = df["Price"].astype(float).values

split = pd.read_csv("gold_split.csv")
train_size = int(split.loc[0, "train_size"])
n_test = len(price) - train_size
print(f"Loaded {len(price)} days | train_size = {train_size} | test = {n_test} days")

# Model on the LOG scale, predicting the daily LOG RETURN (matches lambda = 0)
logp = np.log(price)
ret  = np.diff(logp)                          # ret[t] = logp[t+1] - logp[t]

def make_lag_matrix(series, n_lags, end):
    """Rows whose TARGET index is < `end` (so we never use future info)."""
    X, y = [], []
    for t in range(n_lags, end):
        X.append(series[t - n_lags:t]); y.append(series[t])
    return np.array(X), np.array(y)

# The return that produces price[i] is ret[i-1]. First test price is
# price[train_size], produced by ret[train_size-1]. So test return targets are
# indices train_size-1 .. len(ret)-1.
first_test_ret = train_size - 1
actual_levels  = price[train_size:]           # the real test prices

# ---------------------------------------------------------------------------
# 2. ONE-STEP-AHEAD WALK-FORWARD for the tree models
#    For each test day: train on everything up to "now", predict the next
#    day's return using the TRUE previous 20 returns, convert to a price.
#    (Models are refit periodically, not every single day, to save time —
#     refitting daily gives essentially identical results but is much slower.)
# ---------------------------------------------------------------------------
REFIT_EVERY = 20   # refit every 20 steps (set to 1 for daily refit)

def walk_forward_tree(make_model):
    preds = []; model = None
    for step, t in enumerate(range(first_test_ret, len(ret))):
        if step % REFIT_EVERY == 0:                  # (re)train on data up to now
            X_tr, y_tr = make_lag_matrix(ret, N_LAGS, end=t)
            model = make_model(); model.fit(X_tr, y_tr)
        x = ret[t - N_LAGS:t].reshape(1, -1)          # TRUE previous 20 returns
        r_hat = model.predict(x)[0]
        preds.append(np.exp(logp[t] + r_hat))         # price = exp(last log price + predicted return)
    return np.array(preds)

rf_levels = walk_forward_tree(
    lambda: RandomForestRegressor(n_estimators=300, random_state=42))
xgb_levels = walk_forward_tree(
    lambda: XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3,
                         random_state=42, verbosity=0))

# ---------------------------------------------------------------------------
# 3. ONE-STEP-AHEAD WALK-FORWARD for the LSTM (small, fair, scaled, fixed seed)
# ---------------------------------------------------------------------------
try:
    import tensorflow as tf
    tf.random.set_seed(42)
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Input

    def build_lstm():
        m = Sequential([Input((N_LAGS, 1)), LSTM(16), Dense(1)])
        m.compile(optimizer="adam", loss="mse")
        return m

    lstm_preds = []; scaler = None; lstm = None
    LSTM_REFIT_EVERY = 40   # LSTMs are slow to train; refit less often
    for step, t in enumerate(range(first_test_ret, len(ret))):
        if step % LSTM_REFIT_EVERY == 0:
            X_tr, y_tr = make_lag_matrix(ret, N_LAGS, end=t)
            scaler = StandardScaler()
            X_tr_s = scaler.fit_transform(X_tr).reshape(-1, N_LAGS, 1)
            lstm = build_lstm()
            lstm.fit(X_tr_s, y_tr, epochs=30, batch_size=32, verbose=0)
        x = scaler.transform(ret[t - N_LAGS:t].reshape(1, -1)).reshape(1, N_LAGS, 1)
        r_hat = float(lstm.predict(x, verbose=0)[0, 0])
        lstm_preds.append(np.exp(logp[t] + r_hat))
    lstm_levels = np.array(lstm_preds)
    have_lstm = True
except Exception as e:
    print(f"\n(LSTM skipped — {type(e).__name__}: {e})")
    have_lstm = False

# Naive one-step random walk: predict each day = the actual PREVIOUS day's price.
naive_levels = price[train_size - 1:len(price) - 1]

# ---------------------------------------------------------------------------
# 4. Combined accuracy table
# ---------------------------------------------------------------------------
def metrics(a, p):
    return (round(np.sqrt(mean_squared_error(a, p)), 4),
            round(mean_absolute_error(a, p), 4),
            round(np.mean(np.abs((a - p) / a)) * 100, 4))

rows = []
try:
    cls = pd.read_csv("gold_classical_forecasts.csv")   # one-step ARIMA & ETS from R
    for col in ["ARIMA", "ETS"]:
        rows.append([f"{col} (R)", *metrics(cls["Actual"].values, cls[col].values)])
except FileNotFoundError:
    print("(gold_classical_forecasts.csv not found — re-run the R script.)")

ml_preds = [("Random Forest", rf_levels), ("XGBoost", xgb_levels)]
if have_lstm:
    ml_preds.append(("LSTM", lstm_levels))
ml_preds.append(("Naive (RW)", naive_levels))
for name, p in ml_preds:
    rows.append([name, *metrics(actual_levels, p)])

results = (pd.DataFrame(rows, columns=["Model", "RMSE", "MAE", "MAPE"])
             .sort_values("RMSE").reset_index(drop=True))
print("\n=== ONE-STEP-AHEAD ACCURACY ON TEST 20% (lower = better) ===")
print(results.to_string(index=False))

# ---------------------------------------------------------------------------
# 5. Diebold-Mariano test (H0: equal accuracy; p < 0.05 = a real difference)
# ---------------------------------------------------------------------------
def diebold_mariano(actual, p1, p2):
    d = (actual - p1)**2 - (actual - p2)**2
    T = len(d); g0 = np.var(d, ddof=0)
    dm = d.mean() / np.sqrt(g0 / T) if g0 > 0 else 0.0
    return round(dm, 3), round(2 * (1 - norm.cdf(abs(dm))), 4)

preds = {}
try:
    preds["ARIMA"] = cls["ARIMA"].values
    preds["ETS"]   = cls["ETS"].values
except Exception:
    pass
preds.update({"Random Forest": rf_levels, "XGBoost": xgb_levels, "Naive (RW)": naive_levels})
if have_lstm:
    preds["LSTM"] = lstm_levels

best = results.loc[0, "Model"].replace(" (R)", "")
base = preds.get(best)
print(f"\n=== DIEBOLD-MARIANO vs best model ({best}) ===")
if base is not None:
    for name, p in preds.items():
        if name == best:
            continue
        dm, pv = diebold_mariano(actual_levels, base, p)
        print(f"  {best:>13} vs {name:<14} DM={dm:>7}  p={pv:<7} -> "
              f"{'different' if pv < 0.05 else 'NOT significantly different (tied)'}")
print("\np > 0.05 means the best model is NOT provably better than that rival.")

# ---------------------------------------------------------------------------
# 6. Save
# ---------------------------------------------------------------------------
out = {"Date": df["Date"].values[train_size:], "Actual": actual_levels,
       "RF": rf_levels, "XGB": xgb_levels, "Naive": naive_levels}
if have_lstm:
    out["LSTM"] = lstm_levels
pd.DataFrame(out).to_csv("ml_gold_partB_forecast.csv", index=False)
print("\nSaved -> ml_gold_partB_forecast.csv")
# ---------------------------------------------------------------------------
# 7. VALIDATION ADDITIONS
# ---------------------------------------------------------------------------

# ── 7A: Train-vs-Test Overfitting Check ──
# Refit one final model on all training data and compare
# train error vs test error. Large gap = overfitting.

print("\n=== OVERFITTING CHECK (Train vs Test Error) ===")
X_train_full, y_train_full = make_lag_matrix(ret, N_LAGS, end=first_test_ret)
X_test_full,  y_test_full  = make_lag_matrix(ret, N_LAGS, end=len(ret))
X_test_only   = X_test_full[len(X_train_full):]
y_test_only   = y_test_full[len(y_train_full):]

for name, make_model in [
    ("Random Forest",
     lambda: RandomForestRegressor(n_estimators=300, random_state=42)),
    ("XGBoost",
     lambda: XGBRegressor(n_estimators=300, learning_rate=0.05,
                          max_depth=3, random_state=42, verbosity=0))
]:
    m = make_model()
    m.fit(X_train_full, y_train_full)

    train_rmse = np.sqrt(mean_squared_error(
        y_train_full, m.predict(X_train_full)))
    test_rmse  = np.sqrt(mean_squared_error(
        y_test_only,  m.predict(X_test_only)))
    ratio = test_rmse / train_rmse

    print(f"\n{name}:")
    print(f"  Train RMSE (log-return scale): {train_rmse:.6f}")
    print(f"  Test  RMSE (log-return scale): {test_rmse:.6f}")
    print(f"  Test/Train ratio             : {ratio:.3f}")
    print(f"  {'ACCEPTABLE — ratio near 1, no severe overfitting' if ratio < 2.0 else 'WARNING — possible overfitting, ratio > 2'}")

print("\nNote: Ratio near 1.0 = model generalises well.")
print("Walk-forward evaluation already guards against look-ahead bias.")

# ── 7B: Feature Importance (RF and XGBoost) ──
print("\n=== FEATURE IMPORTANCE ===")
print("Lag 1 = yesterday's return, Lag 20 = return 20 days ago")

rf_final = RandomForestRegressor(n_estimators=300, random_state=42)
rf_final.fit(X_train_full, y_train_full)

xgb_final = XGBRegressor(n_estimators=300, learning_rate=0.05,
                          max_depth=3, random_state=42, verbosity=0)
xgb_final.fit(X_train_full, y_train_full)

lag_names = [f"Lag_{i}" for i in range(1, N_LAGS + 1)]

rf_imp = pd.Series(rf_final.feature_importances_,
                   index=lag_names).sort_values(ascending=False)
xgb_imp = pd.Series(xgb_final.feature_importances_,
                    index=lag_names).sort_values(ascending=False)

print("\nRandom Forest — Top 5 Features:")
print(rf_imp.head(5).round(4).to_string())

print("\nXGBoost — Top 5 Features:")
print(xgb_imp.head(5).round(4).to_string())

feat_imp_df = pd.DataFrame({
    "Lag"          : lag_names,
    "RF_Importance": rf_final.feature_importances_.round(6),
    "XGB_Importance": xgb_final.feature_importances_.round(6)
}).sort_values("RF_Importance", ascending=False)

feat_imp_df.to_csv("gold_feature_importance.csv", index=False)
print("\nSaved -> gold_feature_importance.csv")

# ── 7C: Methodology Framing (printed for report reference) ──
print("\n=== METHODOLOGY NOTES FOR REPORT ===")
print("""
Walk-forward evaluation (Hyndman & Athanasopoulos, 2021, Ch.5):
  Each test observation is predicted one step ahead using only
  past data — equivalent to time-series cross-validation.
  This removes multi-step drift bias and gives a fair comparison
  with one-step-ahead classical forecasts (ARIMA, ETS).

Hyperparameter choices:
  Random Forest : n_estimators=300, no depth limit (avoids underfitting)
  XGBoost       : n_estimators=300, lr=0.05, max_depth=3 (conservative
                  depth prevents overfitting on financial returns)
  LSTM          : 1 layer x 16 units, 30 epochs (small by design —
                  larger networks overfit on <2000 daily observations)
  N_LAGS=20     : approximately one trading month of history,
                  consistent with short-memory assumption of daily returns.

Prediction intervals and classification metrics are outside
the scope of this point-forecast comparison study.
""")























# 8. LSTM VALIDATION DIAGNOSTICS
# ---------------------------------------------------------------------------

if have_lstm:

    print("\n=== LSTM VALIDATION DIAGNOSTICS ===")

    from statsmodels.stats.diagnostic import acorr_ljungbox
    import matplotlib.pyplot as plt

    # ---------------------------------------------------------------
    # 8A. Train vs Test RMSE (Overfitting Check)
    # ---------------------------------------------------------------

    scaler_diag = StandardScaler()

    X_train_lstm = scaler_diag.fit_transform(
        X_train_full
    ).reshape(-1, N_LAGS, 1)

    X_test_lstm = scaler_diag.transform(
        X_test_only
    ).reshape(-1, N_LAGS, 1)

    lstm_diag = build_lstm()

    history = lstm_diag.fit(
        X_train_lstm,
        y_train_full,
        epochs=30,
        batch_size=32,
        validation_split=0.20,
        verbose=0
    )

    train_pred = lstm_diag.predict(
        X_train_lstm,
        verbose=0
    ).flatten()

    test_pred = lstm_diag.predict(
        X_test_lstm,
        verbose=0
    ).flatten()

    train_rmse = np.sqrt(
        mean_squared_error(y_train_full, train_pred)
    )

    test_rmse = np.sqrt(
        mean_squared_error(y_test_only, test_pred)
    )

    ratio = test_rmse / train_rmse

    print("\nLSTM Overfitting Check:")
    print(f"  Train RMSE : {train_rmse:.6f}")
    print(f"  Test RMSE  : {test_rmse:.6f}")
    print(f"  Ratio      : {ratio:.3f}")

    print(
        "  ACCEPTABLE — no severe overfitting"
        if ratio < 2
        else
        "  WARNING — possible overfitting"
    )

    # ---------------------------------------------------------------
    # 8B. Learning Curve
    # ---------------------------------------------------------------

    plt.figure(figsize=(8,4))

    plt.plot(
        history.history["loss"],
        label="Training Loss"
    )

    plt.plot(
        history.history["val_loss"],
        label="Validation Loss"
    )

    plt.title("Gold LSTM Learning Curve")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.legend()

    plt.tight_layout()
    plt.savefig("gold_lstm_learning_curve.png", dpi=300)
    plt.close()

    print("\nSaved -> gold_lstm_learning_curve.png")

    # ---------------------------------------------------------------
    # 8C. Forecast Residual Diagnostics
    # ---------------------------------------------------------------

    residuals = actual_levels - lstm_levels

    print("\nResidual Summary:")
    print(f"  Mean Residual : {np.mean(residuals):.6f}")
    print(f"  Std Residual  : {np.std(residuals):.6f}")

    # Residual time plot

    plt.figure(figsize=(8,4))

    plt.plot(
        df["Date"].values[train_size:],
        residuals
    )

    plt.axhline(
        0,
        linestyle="--"
    )

    plt.title("Gold LSTM Forecast Residuals")
    plt.xlabel("Date")
    plt.ylabel("Residual")

    plt.tight_layout()
    plt.savefig("gold_lstm_residuals.png", dpi=300)
    plt.close()

    print("Saved -> gold_lstm_residuals.png")

    # Residual histogram

    plt.figure(figsize=(7,4))

    plt.hist(
        residuals,
        bins=30
    )

    plt.title("Gold LSTM Residual Distribution")
    plt.xlabel("Residual")
    plt.ylabel("Frequency")

    plt.tight_layout()
    plt.savefig("gold_lstm_residual_histogram.png", dpi=300)
    plt.close()

    print("Saved -> gold_lstm_residual_histogram.png")

    # ---------------------------------------------------------------
    # 8D. Ljung-Box Test
    # ---------------------------------------------------------------

    lb = acorr_ljungbox(
        residuals,
        lags=[10,20],
        return_df=True
    )

    print("\nLjung-Box Test on LSTM Residuals:")
    print(lb.round(4))

    if (lb["lb_pvalue"] > 0.05).all():
        print(
            "\nResult: Residuals resemble white noise "
            "(no significant autocorrelation)."
        )
    else:
        print(
            "\nResult: Some residual autocorrelation remains."
        )

    print("\nLSTM diagnostics completed.")

Loaded 1537 days | train_size = 1229 | test = 308 days

=== ONE-STEP-AHEAD ACCURACY ON TEST 20% (lower = better) ===
        Model      RMSE       MAE   MAPE
    ARIMA (R) 2642.1797 1394.6916 1.0850
   Naive (RW) 2642.1797 1394.6916 1.0850
      ETS (R) 2650.1559 1384.9578 1.0775
Random Forest 2683.3595 1454.3337 1.1412
         LSTM 2872.4072 1561.8557 1.2063
      XGBoost 3023.4306 1544.4247 1.1965

=== DIEBOLD-MARIANO vs best model (ARIMA) ===
          ARIMA vs ETS            DM= -0.379  p=0.7049  -> NOT significantly different (tied)
          ARIMA vs Random Forest  DM= -0.722  p=0.47    -> NOT significantly different (tied)
          ARIMA vs XGBoost        DM= -2.071  p=0.0383  -> different
          ARIMA vs Naive (RW)     DM=    0.0  p=1.0     -> NOT significantly different (tied)
          ARIMA vs LSTM           DM= -2.155  p=0.0312  -> different

p > 0.05 means the best model is NOT provably better than that rival.

Saved -> ml_gold_partB_forecast.csv

=== OVERFITTING CHEC